# 資料を調べて答えるチャットボットを作る (複数ターンの RAG)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/21_dialogue/chat_rag.ipynb)

NLP クックブック 21-08 ／ 対話システム

対話の流れを踏まえながら、手元の資料を検索して答えるチャットボットを作りたい。雑談と資料に基づく回答を両立させたい。

In [1]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

docs = [
    "クックブック食堂の営業時間は平日 11 時から 15 時、土日は 11 時から 20 時です。月曜日は定休日です。",
    "クックブック食堂の看板メニューは、3 日間煮込んだ特製ビーフカレー (950 円) です。大盛りは 100 円増しです。",
    "クックブック食堂は 2019 年に京都市左京区で開業しました。店主の田中さんはインドで 2 年間修行しました。",
    "駐車場は店舗の裏に 3 台分あります。満車の場合は近隣のコインパーキングをご利用ください。",
    "予約は電話 (075-000-0000) のみ受け付けています。5 名以上の場合は前日までにご連絡ください。",
    "支払いは現金と QR コード決済に対応しています。クレジットカードは使えません。",
    "辛さは 5 段階から選べます。お子様向けの甘口カレー (600 円) もあります。",
]
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
doc_emb = embedder.encode(["passage: " + d for d in docs], normalize_embeddings=True)

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def generate(messages, max_new_tokens=80):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [2]:
import re
from transformers import AutoModelForSequenceClassification

nli_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
nli_tok = AutoTokenizer.from_pretrained(nli_name)
nli = AutoModelForSequenceClassification.from_pretrained(nli_name).eval()

def supported(context, reply):
    """応答の各文が資料から言えるか (含意確率の最小値)"""
    sents = [s for s in re.split(r"(?<=[。！？])", reply) if s.strip()]
    with torch.no_grad():
        probs = nli(**nli_tok([context] * len(sents), sents, return_tensors="pt", padding=True, truncation=True)).logits.softmax(-1)
    return probs[:, 0].min().item()

SYSTEM = "あなたはクックブック食堂の案内係です。丁寧な言葉で、1 文で短く答えてください。"
SMALL_TALK = re.compile(r"こんにちは|こんばんは|おはよう|はじめまして|ありがとう|助かりました|さようなら|また来ます")
REFERENCE = re.compile(r"それ|そこ|その|これ|あれ|そちら|ここ")
FALLBACK = "申し訳ありませんが、その件はわかりかねます。お店に直接お問い合わせください。"

class RagChat:
    def __init__(self):
        self.history = []

    def rewrite(self, question):
        """指示語を含む質問だけ、対話の文脈を補って検索用の質問にする"""
        if not self.history or not REFERENCE.search(question):
            return question
        dialogue = "\n".join(f"{'ユーザー' if m['role'] == 'user' else 'アシスタント'}: {m['content']}" for m in self.history[-4:])
        prompt = ("次の対話の最後の質問を、対話を読んでいない人にも意味が通じる、検索用の 1 つの質問に書き換えてください。"
                  f"書き換えた質問だけを出力してください。\n\n{dialogue}\nユーザー: {question}")
        return generate([{"role": "user", "content": prompt}], max_new_tokens=50).strip("「」")

    def say(self, question):
        if SMALL_TALK.search(question):                      # 挨拶やお礼は、検索せずにそのまま応答する
            reply, query, hits, score = generate([{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}], 40), None, [], None
        else:
            query = self.rewrite(question)
            sims = doc_emb @ embedder.encode("query: " + query, normalize_embeddings=True)
            hits = [int(i) for i in sims.argsort()[::-1][:2]]
            context = "\n".join(docs[i] for i in hits)
            # 資料は履歴に残さず、今回の質問にだけ付ける (履歴が資料で膨らむのを防ぐ)
            content = f"次の資料だけに基づいて質問に答えてください。\n\n# 資料\n{context}\n\n# 質問\n{query}"
            reply = generate([{"role": "system", "content": SYSTEM}, {"role": "user", "content": content}], 70)
            score = supported(context, reply)
            if score < 0.5:                                  # 資料から言えないことを述べていたら、定型の応答に差し替える
                reply = FALLBACK
        self.history += [{"role": "user", "content": question}, {"role": "assistant", "content": reply}]
        return reply, query, hits, score

bot = RagChat()
for user in ["こんにちは。", "おすすめのメニューは何ですか？", "それはいくらですか？", "子ども向けのメニューはありますか？",
             "日曜の 18 時は開いていますか？", "クレジットカードは使えますか？", "ペットを連れて行けますか？", "ありがとうございました。"]:
    reply, query, hits, score = bot.say(user)
    detail = "検索なし" if query is None else f"検索クエリ: {query} / 資料: {hits} / 裏付け: {score:.2f}"
    print(f"あなた: {user}\nボット: {reply}\n   ({detail})")

あなた: こんにちは。
ボット: こんにちは。クックブック食堂へようこそ。本日はどのようなお料理をお探しでしょうか？お手伝いさせていただきます。
   (検索なし)
あなた: おすすめのメニューは何ですか？
ボット: クックブック食堂のおすすめメニューは、3 日間煮込んだ特製ビーフカレー（950 円）です。特に辛さを調整できるので、お客様の好みに合った辛さでお楽しみいただけます。また、お子様向けに甘口カレー（600 円）もご用意しております。
   (検索クエリ: おすすめのメニューは何ですか？ / 資料: [1, 6] / 裏付け: 0.93)
あなた: それはいくらですか？
ボット: 申し訳ありませんが、その件はわかりかねます。お店に直接お問い合わせください。
   (検索クエリ: クックブック食堂のおすすめのカレーメニューと価格を教えてください。特に、特製ビーフカレーと甘口カレーの価格に興味があります。 / 資料: [1, 6] / 裏付け: 0.01)
あなた: 子ども向けのメニューはありますか？
ボット: はい、お子様向けの甘口カレーをご用意しております（600円）。
   (検索クエリ: 子ども向けのメニューはありますか？ / 資料: [6, 1] / 裏付け: 1.00)
あなた: 日曜の 18 時は開いていますか？
ボット: はい、日曜の 18 時は開いております。クックブック食堂の土日の営業時間は 11 時から 20 時までとなっております。
   (検索クエリ: 日曜の 18 時は開いていますか？ / 資料: [0, 4] / 裏付け: 0.72)
あなた: クレジットカードは使えますか？
ボット: いいえ、クレジットカードはお使いいただけません。現金または QR コード決済をご利用ください。
   (検索クエリ: クレジットカードは使えますか？ / 資料: [5, 3] / 裏付け: 1.00)
あなた: ペットを連れて行けますか？
ボット: 申し訳ありませんが、その件はわかりかねます。お店に直接お問い合わせください。
   (検索クエリ: ペットを連れて行けますか？ / 資料: [4, 5] / 裏付け: 0.00)
あなた: ありがとうございました。
ボット: どういたしまして、またのご利用をお待ちしております。
   (検索なし)


- 各ターンで「(必要なら) 質問の書き換え → 検索 → 資料を付けて生成 → 資料との照合」を行う。
- 小さな埋め込みモデルの類似度では、挨拶と質問、答えのある質問とない質問をうまく区別できない (挨拶のほうが類似度が高く出ることもある)。そこで、挨拶は規則で判定し、答えが資料にあるかどうかは生成した応答を NLI で資料と照合して確かめている。
- 3B クラスのモデルは、資料にないこと (ペット) を聞かれると、もっともらしい答えを作ってしまうことが多い。照合で裏付けが取れない応答は定型文に差し替えると、誤った案内を防げる。そのかわり、正しい応答まで差し替えてしまうこともある (安全側に倒す設計)。
- 質問の書き換えは、指示語を含むときだけ行う。毎回書き換えると、関係のない前の話題を持ち込んで検索を外すことがある (RAG の章の「質問を書き換える」を参照)。
- 検索した資料は、そのターンの入力にだけ付けて、履歴には質問と回答だけを残す。